# TensorBoard-Plots der MoE-Random-Search

Dieses Notebook liest alle TensorBoard-Scalar-Logs aus `moe_runs/randomsearch_new` und erzeugt PDF-Abbildungen fuer den Anhang der Masterarbeit:

- Trainings- und Validierungsgenauigkeit mit Hard-Routing
- Nutzung der Klassifikatoren bei Hard-Routing
- Validierungsverlaeufe fuer Top-k-Routing


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, MultipleLocator, PercentFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Einheitlicher Plot-Stil der Masterarbeit.
plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
})

# Plot 1 und 3 werden mit 0.7\linewidth eingebunden,
# Plot 2 über die volle Textbreite.
FIGSIZE_07 = (3.4, 2.3)
FIGSIZE_FULL = (6.8, 2.3)


In [ ]:
def finde_moe_root() -> Path:
    # Findet den moe-Ordner, egal ob das Notebook aus dem Repo-Root oder aus moe/plots gestartet wird.
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for kandidat in (base, base / "moe", base / "single_pulse_classifier_training" / "moe"):
            if (kandidat / "moe_runs" / "randomsearch_new").exists():
                return kandidat
    raise FileNotFoundError("moe_runs/randomsearch_new konnte vom aktuellen Arbeitsverzeichnis aus nicht gefunden werden.")


MOE_ROOT = finde_moe_root()
RUN_ROOT = MOE_ROOT / "moe_runs" / "randomsearch_new"
OUT_DIR = MOE_ROOT / "plots" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MOE_ROOT, RUN_ROOT, OUT_DIR

In [ ]:
def dekodiere_float_token(token: str) -> str:
    return token.replace("p", ".").replace("em", "e-")


def parse_run_name(run_name: str) -> dict:
    muster = {
        "trial": r"trial(\d+)",
        "seed": r"seed(\d+)",
        "expert_lr": r"expertlr([^_]+)",
        "rejector_lr": r"rejectorlr([^_]+)",
        "weight_decay": r"wd([^_]+)",
        "temperatur": r"temp([^_]+)",
    }
    meta = {"run": run_name}
    for key, pattern in muster.items():
        match = re.search(pattern, run_name)
        if not match:
            continue
        wert = match.group(1)
        if key in {"trial", "seed"}:
            meta[key] = int(wert)
        else:
            meta[key] = dekodiere_float_token(wert)

    trial = meta.get("trial", "?")
    seed = meta.get("seed", "?")
    temperatur = meta.get("temperatur", "?")
    meta["label"] = f"Durchlauf {trial} (Seed {seed}, T={temperatur})"
    return meta


def load_tensorboard_scalars(run_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    zeilen = []
    run_zeilen = []

    for run_dir in sorted(p for p in run_root.iterdir() if p.is_dir()):
        tb_dir = run_dir / "tensorboard"
        meta = parse_run_name(run_dir.name)
        meta.update({"path": str(run_dir), "tensorboard_path": str(tb_dir)})

        if not tb_dir.exists():
            meta.update({"status": "TensorBoard-Ordner fehlt", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        accumulator = EventAccumulator(str(tb_dir), size_guidance={"scalars": 0})
        try:
            accumulator.Reload()
        except Exception as exc:
            meta.update({"status": f"Lesefehler: {exc}", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        tags = accumulator.Tags().get("scalars", [])
        n_points = 0
        for tag in tags:
            events = accumulator.Scalars(tag)
            n_points += len(events)
            for event in events:
                zeilen.append({
                    **meta,
                    "tag": tag,
                    "epoche": event.step,
                    "wall_time": event.wall_time,
                    "wert": float(event.value),
                })

        meta.update({"status": "ok" if tags else "keine Scalar-Tags", "n_tags": len(tags), "n_points": n_points})
        run_zeilen.append(meta)

    scalars = pd.DataFrame(zeilen)
    runs = pd.DataFrame(run_zeilen).sort_values(["status", "trial", "seed", "run"], na_position="last")
    return scalars, runs


scalars, runs = load_tensorboard_scalars(RUN_ROOT)
print(f"Geladen: {len(scalars):,} Scalar-Punkte aus {runs.query('n_points > 0').shape[0]} Durchlaufs.")
display(runs[["label", "status", "n_tags", "n_points", "run"]])

In [ ]:
ERWARTETE_TAGS = [
    "train/accuracy",
    "val_hard/accuracy",
    "val_hard/usage_small",
    "val_hard/usage_mid",
    "val_hard/usage_large",
    "val_topk/accuracy",
]

verfuegbare_tags = sorted(scalars["tag"].unique()) if not scalars.empty else []
fehlende_tags = [tag for tag in ERWARTETE_TAGS if tag not in verfuegbare_tags]
if fehlende_tags:
    warnings.warn("Folgende erwartete TensorBoard-Tags fehlen: " + ", ".join(fehlende_tags))

print("Verfuegbare Scalar-Tags:")
for tag in verfuegbare_tags:
    print(" -", tag)

In [ ]:
gueltige_runs = runs.loc[runs["n_points"] > 0].copy()
gueltige_runs = gueltige_runs.sort_values(["trial", "seed", "temperatur", "run"], na_position="last")
run_order = gueltige_runs["run"].tolist()
label_by_run = dict(zip(gueltige_runs["run"], gueltige_runs["label"]))

farben = plt.colormaps["tab20"].resampled(max(len(run_order), 1))
color_by_run = {run: farben(i) for i, run in enumerate(run_order)}

USAGE_TAGS = {
    "val_hard/usage_small": r"$f_{\mathrm{small}}$",
    "val_hard/usage_mid": r"$f_{\mathrm{mid}}$",
    "val_hard/usage_large": r"$f_{\mathrm{large}}$",
}

TOPK_TAG_LABELS = {
    "val_topk/accuracy": "Genauigkeit",
    "val_topk/total": "Gesamt-Loss",
    "val_topk/ensemble": "Ensemble-Loss",
    "val_topk/expert_small": r"Loss $f_{small}$",
    "val_topk/expert_mid": r"Loss $f_{mid}$",
    "val_topk/expert_large": r"Loss $f_{large}$",
    "val_topk/usage_small": r"Nutzung $f_{small}$",
    "val_topk/usage_mid": r"Nutzung $f_{mid}$",
    "val_topk/usage_large": r"Nutzung $f_{large}$",
}


def tag_frame(tag: str) -> pd.DataFrame:
    frame = scalars.loc[scalars["tag"] == tag, ["run", "label", "epoche", "wert"]].copy()
    return frame.sort_values(["run", "epoche"])


def ist_anteil(frame: pd.DataFrame) -> bool:
    if frame.empty:
        return False
    return frame["wert"].dropna().between(-0.02, 1.02).all()


def style_axis(ax, ylabel=None, ylim=None, ytick_step=None, percent=False):
    ax.set_xlabel("Epoche", labelpad=2)
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=2)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if ytick_step is not None:
        ax.yaxis.set_major_locator(MultipleLocator(ytick_step))
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.grid(axis="both", color="0.90", linewidth=0.7, linestyle="-")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def speichere_abbildung(fig, dateiname: str):
    path = OUT_DIR / f"{dateiname}.pdf"
    fig.savefig(path)
    print(f"Gespeichert: {path.relative_to(MOE_ROOT)}")


def beste_runs_nach_tag(tag: str) -> list[str]:
    frame = tag_frame(tag)
    if frame.empty:
        return run_order
    ranking = (
        frame.groupby("run", as_index=False)["wert"]
        .max()
        .sort_values("wert", ascending=False)
    )
    return ranking["run"].tolist()

## Trainings-Genauigkeit und Validierungsgenauigkeit mit Hard-Routing

In [ ]:
fig, ax = plt.subplots(
    figsize=FIGSIZE_07,
    constrained_layout=True,
)

run_order_accuracy = beste_runs_nach_tag("val_hard/accuracy")

for run in run_order_accuracy:
    train_frame = tag_frame("train/accuracy").loc[lambda df: df["run"] == run]
    val_frame = tag_frame("val_hard/accuracy").loc[lambda df: df["run"] == run]
    color = color_by_run[run]

    if not train_frame.empty:
        ax.plot(
            train_frame["epoche"],
            train_frame["wert"],
            color=color,
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
            zorder=1,
        )

    if not val_frame.empty:
        ax.plot(
            val_frame["epoche"],
            val_frame["wert"],
            color=color,
            linestyle="-",
            linewidth=0.9,
            alpha=0.85,
            zorder=2,
        )

style_axis(
    ax,
    ylabel="Genauigkeit",
    ylim=(0.50, 1.00),
    ytick_step=0.10,
    percent=True,
)

style_handles = [
    Line2D([0], [0], color="0.25", linestyle="--", linewidth=0.9, label="Training"),
    Line2D([0], [0], color="0.25", linestyle="-", linewidth=0.9, label="Validierung"),
]
ax.legend(
    handles=style_handles,
    loc="lower right",
    frameon=True,
    framealpha=0.9,
    borderpad=0.3,
    labelspacing=0.25,
    handlelength=1.8,
)

speichere_abbildung(fig, "expertanalysis_train_val_hard_accuracy")
plt.show()


## Nutzung der Klassifikatoren bei Hard-Routing

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=FIGSIZE_FULL,
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

run_order_usage = beste_runs_nach_tag("val_hard/accuracy")

for ax, (tag, title) in zip(axes, USAGE_TAGS.items()):
    frame = tag_frame(tag)

    for run in run_order_usage:
        run_frame = frame.loc[frame["run"] == run]
        if run_frame.empty:
            continue

        ax.plot(
            run_frame["epoche"],
            run_frame["wert"],
            color=color_by_run[run],
            linewidth=0.9,
            alpha=0.85,
        )

    ax.set_title(title, pad=3)
    style_axis(
        ax,
        ylabel="Anteil" if ax is axes[0] else None,
        ylim=(0.0, 1.0),
        ytick_step=0.20,
        percent=True,
    )

speichere_abbildung(fig, "expertanalysis_val_hard_classifier_usage")
plt.show()


## Validierungsverläufe für Top-k-Routing

In [ ]:
fig, ax = plt.subplots(
    figsize=FIGSIZE_07,
    constrained_layout=True,
)

frame = tag_frame("val_topk/accuracy")
run_order_topk = beste_runs_nach_tag("val_topk/accuracy")

for run in run_order_topk:
    run_frame = frame.loc[frame["run"] == run]
    if run_frame.empty:
        continue

    ax.plot(
        run_frame["epoche"],
        run_frame["wert"],
        color=color_by_run[run],
        linewidth=0.9,
        alpha=0.85,
    )

style_axis(
    ax,
    ylabel="Validierungsgenauigkeit",
    ylim=(0.60, 0.90),
    ytick_step=0.05,
    percent=True,
)

speichere_abbildung(fig, "expertanalysis_val_topk_accuracy")
plt.show()
